# AI_CLASSIFY — Order Comment Classification

## Use Case Overview

`AI_CLASSIFY` labels each row of text into one of your predefined categories using an LLM. Unlike rule-based keyword matching, it understands intent and context — so "this arrived broken" maps to *damage* even without the word "damage".

**SA use case:** Automatically tag unstructured text columns (support tickets, order notes, survey responses) without writing a regex or training a model.

**Dataset:** TPC-H `ORDERS.O_COMMENT` — free-text comments attached to each order (~1.5M rows). We sample 500 rows for demo speed.

**What we'll classify:**
- `urgent` — time-sensitive requests
- `billing` — pricing or payment issues  
- `shipping` — delivery or logistics concerns
- `quality` — product condition or defect mentions
- `general` — everything else

## Step 1 — Setup & Connect

In [ ]:
import os
import pandas as pd
import snowflake.connector

conn = snowflake.connector.connect(
    connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME") or "SFCOGSOPS-SNOWHOUSE_AWS_US_WEST_2"
)
conn.cursor().execute("USE DATABASE GENAI_LEARNING")
conn.cursor().execute("USE SCHEMA PUBLIC")
print("Connected:", conn.account)

## Step 2 — Data Exploration

Preview the order comments we'll be classifying.

In [ ]:
sample_df = pd.read_sql("""
    SELECT order_key, order_date, order_status, total_price, order_comment
    FROM order_comments_vw
    SAMPLE (500 ROWS)
""", conn)
print(f"Sample size: {len(sample_df)} rows")
sample_df[["order_key", "order_comment"]].head(10)

## Step 3 — Classify with AI_CLASSIFY

`AI_CLASSIFY(value, categories)` — the function takes:
- `value`: the text to classify (a column or expression)
- `categories`: an ARRAY of label strings

It returns a VARIANT with `label` (the predicted class) and `score` (confidence 0–1).

We create a temporary table from our sample, then classify all rows in a single SQL statement.

In [ ]:
conn.cursor().execute("""
    CREATE OR REPLACE TEMPORARY TABLE order_sample AS
    SELECT order_key, order_date, order_status, total_price, order_comment
    FROM order_comments_vw
    SAMPLE (500 ROWS)
""")

classified_df = pd.read_sql("""
    SELECT
        order_key,
        order_comment,
        order_status,
        total_price,
        AI_CLASSIFY(order_comment, ['urgent','billing','shipping','quality','general'])
            :label::STRING  AS category,
        AI_CLASSIFY(order_comment, ['urgent','billing','shipping','quality','general'])
            :score::FLOAT   AS confidence
    FROM order_sample
""", conn)
classified_df.head(10)

## Step 4 — Analyse the Distribution

Look at how comments are distributed across categories and the average confidence per label.

In [ ]:
dist = classified_df.groupby("category").agg(
    count=("order_key", "count"),
    avg_confidence=("confidence", "mean"),
    avg_order_value=("total_price", "mean")
).sort_values("count", ascending=False)

print(dist.to_string())
dist["count"].plot(kind="bar", title="Order Comments by Category", ylabel="Count", rot=0)

## Step 5 — Interpretation

**Reading confidence scores:**
- Score > 0.8 → high confidence, safe to act on automatically
- Score 0.5–0.8 → moderate, worth human review for high-value decisions
- Score < 0.5 → ambiguous text; consider broadening or merging categories

**SA tips:**
- You can add descriptions to categories using a list of dicts: `[{'label':'urgent','description':'Time-sensitive or escalated requests'}]`
- `AI_CLASSIFY` is ideal for enriching existing tables as a `DEFAULT` column expression or in a Dynamic Table pipeline
- Combine with `AI_FILTER` to first narrow to actionable rows, then classify only those